In [ ]:
import os
DATA_DIR = os.getenv('DATA_DIR', '/path/to/local/private/data')
SYS_ORDER_FILE = os.getenv('SYS_ORDER_FILE', 'private_sys_order.xlsx')
GEO_FILE = os.getenv('GEO_FILE', 'private_geo.xlsx')
GOOGLE_MAPS_KEY = os.getenv('GOOGLE_MAPS_KEY', '')


In [ ]:
%pip install pandas numpy matplotlib seaborn scikit-learn requests folium openpyxl

In [ ]:
import pandas as pd

# Load the datasets
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train_df.csv'))
test_df = pd.read_csv(os.path.join(DATA_DIR, 'test_df.csv'))

# Display the first few rows of the train dataset
train_df.head()


In [ ]:
train_df.info()

In [ ]:
import pandas as pd
from sklearn.cluster import DBSCAN
import numpy as np

# Load the provided datasets
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train_df.csv'))
test_df = pd.read_csv(os.path.join(DATA_DIR, 'test_df.csv'))

# Convert relevant columns to datetime
train_df['Order_Date'] = pd.to_datetime(train_df['Order_Date'], dayfirst=True, errors='coerce')
train_df['Time_Order_picked'] = pd.to_datetime(train_df['Time_Order_picked'], format='%H:%M:%S', errors='coerce').dt.time
train_df['DateTime_Order_picked'] = pd.to_datetime(train_df['Order_Date'].astype(str) + ' ' + train_df['Time_Order_picked'].astype(str), errors='coerce')

test_df['Order_Date'] = pd.to_datetime(test_df['Order_Date'], dayfirst=True, errors='coerce')
test_df['Time_Order_picked'] = pd.to_datetime(test_df['Time_Order_picked'], format='%H:%M:%S', errors='coerce').dt.time
test_df['DateTime_Order_picked'] = pd.to_datetime(test_df['Order_Date'].astype(str) + ' ' + test_df['Time_Order_picked'].astype(str), errors='coerce')

# Assuming a column that indicates pallets or using an alternative column for computing 'total_full_pallets'
if 'total_full_pallets' not in train_df.columns:
    train_df['total_full_pallets'] = train_df['Time_taken(min)']  # Example alternative

if 'total_full_pallets' not in test_df.columns:
    test_df['total_full_pallets'] = test_df['Time_taken(min)']  # Example alternative

# Optimized function to find consolidation opportunities using clustering
def find_consolidation_opportunities_optimized(df, eps=0.05, min_samples=2, time_window='1H'):
    locations = df[['Delivery_location_latitude', 'Delivery_location_longitude']].to_numpy()
    clustering = DBSCAN(eps=eps, min_samples=min_samples).fit(locations)
    df['cluster'] = clustering.labels_
    
    consolidation_opportunities = []
    
    for cluster_id in set(clustering.labels_):
        if cluster_id == -1:
            continue
        cluster_df = df[df['cluster'] == cluster_id].sort_values(by='DateTime_Order_picked')
        current_group = []
        for i in range(len(cluster_df)):
            current_order = cluster_df.iloc[i]
            if not current_group:
                current_group.append(current_order['ID'])
            else:
                previous_order = cluster_df.loc[cluster_df['ID'] == current_group[-1]].iloc[0]
                time_diff = (current_order['DateTime_Order_picked'] - previous_order['DateTime_Order_picked']).total_seconds() / 3600
                if time_diff <= pd.to_timedelta(time_window).total_seconds() / 3600:
                    combined_pallets = current_order['total_full_pallets'] + previous_order['total_full_pallets']
                    distance = ((current_order['Delivery_location_latitude'] - previous_order['Delivery_location_latitude'])**2 + 
                                (current_order['Delivery_location_longitude'] - previous_order['Delivery_location_longitude'])**2)**0.5
                    consolidation_opportunities.append({
                        'Order 1': current_order['ID'],
                        'Order 2': previous_order['ID'],
                        'Combined Pallets': combined_pallets,
                        'Distance (km)': distance
                    })
    return consolidation_opportunities

# Apply the function to both train and test datasets
train_consolidation_opportunities_optimized = find_consolidation_opportunities_optimized(train_df)
test_consolidation_opportunities_optimized = find_consolidation_opportunities_optimized(test_df)

# Function to create a cleaner format for the consolidation opportunities
def create_clean_format(consolidation_opportunities):
    formatted_data = []
    for opportunity in consolidation_opportunities:
        formatted_data.append({
            'Order 1': opportunity['Order 1'],
            'Order 2': opportunity['Order 2'],
            'Combined Pallets': opportunity['Combined Pallets'],
            'Distance (km)': opportunity['Distance (km)']
        })
    return pd.DataFrame(formatted_data)

# Convert the opportunities to a cleaner format
train_consolidation_cleaned = create_clean_format(train_consolidation_opportunities_optimized)
test_consolidation_cleaned = create_clean_format(test_consolidation_opportunities_optimized)

# Save the cleaned data to Excel files
train_consolidation_cleaned.to_excel(os.path.join(DATA_DIR, 'train_consolidation_cleaned.xlsx'), index=False)
test_consolidation_cleaned.to_excel(os.path.join(DATA_DIR, 'test_consolidation_cleaned.xlsx'), index=False)

# Provide file paths for download
train_file_path_cleaned = os.path.join(DATA_DIR, 'train_consolidation_cleaned.xlsx')
test_file_path_cleaned = os.path.join(DATA_DIR, 'test_consolidation_cleaned.xlsx')

(train_file_path_cleaned, test_file_path_cleaned)


In [ ]:
# # Convert Order_Date and Time_Order_picked to datetime for both datasets
# train_df['Order_Date'] = pd.to_datetime(train_df['Order_Date'], format='%d-%m-%Y')
# train_df['Time_Order_picked'] = pd.to_datetime(train_df['Time_Order_picked'], format='%H:%M:%S').dt.time
# train_df['DateTime_Order_picked'] = pd.to_datetime(train_df['Order_Date'].astype(str) + ' ' + train_df['Time_Order_picked'].astype(str))

# test_df['Order_Date'] = pd.to_datetime(test_df['Order_Date'], format='%d-%m-%Y')
# test_df['Time_Order_picked'] = pd.to_datetime(test_df['Time_Order_picked'], format='%H:%M:%S').dt.time
# test_df['DateTime_Order_picked'] = pd.to_datetime(test_df['Order_Date'].astype(str) + ' ' + test_df['Time_Order_picked'].astype(str))

# # Define a function to find consolidation opportunities
# def find_consolidation_opportunities(df, time_window='1H', distance_threshold=5):
#     # Sort by DateTime_Order_picked
#     df = df.sort_values(by='DateTime_Order_picked')
    
#     # Initialize an empty list to store consolidation opportunities
#     consolidation_opportunities = []
    
#     # Iterate over each order
#     for i in range(len(df)):
#         current_order = df.iloc[i]
#         potential_group = [current_order['ID']]
        
#         # Check for orders within the specified time window and distance threshold
#         for j in range(i+1, len(df)):
#             next_order = df.iloc[j]
#             time_diff = (next_order['DateTime_Order_picked'] - current_order['DateTime_Order_picked']).total_seconds() / 3600  # Convert to hours
#             if time_diff <= pd.to_timedelta(time_window).total_seconds() / 3600:
#                 # Calculate distance between delivery locations (simple Euclidean distance for now)
#                 distance = ((current_order['Delivery_location_latitude'] - next_order['Delivery_location_latitude'])**2 + 
#                             (current_order['Delivery_location_longitude'] - next_order['Delivery_location_longitude'])**2)**0.5
#                 if distance <= distance_threshold:
#                     potential_group.append(next_order['ID'])
#             else:
#                 break
        
#         # If a potential group is found, add to the list
#         if len(potential_group) > 1:
#             consolidation_opportunities.append(potential_group)
    
#     return consolidation_opportunities

# # Apply the function to both train and test datasets
# train_consolidation_opportunities = find_consolidation_opportunities(train_df)
# test_consolidation_opportunities = find_consolidation_opportunities(test_df)

# # Display the consolidation opportunities
# train_consolidation_opportunities, test_consolidation_opportunities


In [ ]:
# from sklearn.cluster import DBSCAN
# import numpy as np

# # Define a function to find consolidation opportunities using clustering
# def find_consolidation_opportunities_optimized(df, eps=0.05, min_samples=2, time_window='1H'):
#     # Convert the delivery locations to a NumPy array
#     locations = df[['Delivery_location_latitude', 'Delivery_location_longitude']].to_numpy()
    
#     # Apply DBSCAN clustering based on location coordinates
#     clustering = DBSCAN(eps=eps, min_samples=min_samples).fit(locations)
#     df['cluster'] = clustering.labels_
    
#     # Initialize an empty list to store consolidation opportunities
#     consolidation_opportunities = []
    
#     # Iterate over each cluster
#     for cluster_id in set(clustering.labels_):
#         if cluster_id == -1:
#             continue
#         cluster_df = df[df['cluster'] == cluster_id]
        
#         # Check for orders within the specified time window
#         cluster_df = cluster_df.sort_values(by='DateTime_Order_picked')
#         current_group = []
#         for i in range(len(cluster_df)):
#             current_order = cluster_df.iloc[i]
#             if not current_group:
#                 current_group.append(current_order['ID'])
#             else:
#                 time_diff = (current_order['DateTime_Order_picked'] - cluster_df.iloc[current_group[-1]]['DateTime_Order_picked']).total_seconds() / 3600
#                 if time_diff <= pd.to_timedelta(time_window).total_seconds() / 3600:
#                     current_group.append(current_order['ID'])
#                 else:
#                     if len(current_group) > 1:
#                         consolidation_opportunities.append(current_group)
#                     current_group = [current_order['ID']]
        
#         if len(current_group) > 1:
#             consolidation_opportunities.append(current_group)
    
#     return consolidation_opportunities

# # Apply the function to both train and test datasets
# train_consolidation_opportunities_optimized = find_consolidation_opportunities_optimized(train_df)
# test_consolidation_opportunities_optimized = find_consolidation_opportunities_optimized(test_df)

# # Display the consolidation opportunities
# train_consolidation_opportunities_optimized, test_consolidation_opportunities_optimized


In [ ]:
# def find_consolidation_opportunities_corrected(df, eps=0.05, min_samples=2, time_window='1H'):
#     # Convert the delivery locations to a NumPy array
#     locations = df[['Delivery_location_latitude', 'Delivery_location_longitude']].to_numpy()
    
#     # Apply DBSCAN clustering based on location coordinates
#     clustering = DBSCAN(eps=eps, min_samples=min_samples).fit(locations)
#     df['cluster'] = clustering.labels_
    
#     # Initialize an empty list to store consolidation opportunities
#     consolidation_opportunities = []
    
#     # Iterate over each cluster
#     for cluster_id in set(clustering.labels_):
#         if cluster_id == -1:
#             continue
#         cluster_df = df[df['cluster'] == cluster_id].sort_values(by='DateTime_Order_picked')
        
#         # Check for orders within the specified time window
#         current_group = []
#         for i in range(len(cluster_df)):
#             current_order = cluster_df.iloc[i]
#             if not current_group:
#                 current_group.append(current_order['ID'])
#             else:
#                 previous_order = cluster_df.iloc[current_group[-1] - 1]  # Reference the previous order in the group
#                 time_diff = (current_order['DateTime_Order_picked'] - previous_order['DateTime_Order_picked']).total_seconds() / 3600
#                 if time_diff <= pd.to_timedelta(time_window).total_seconds() / 3600:
#                     current_group.append(current_order['ID'])
#                 else:
#                     if len(current_group) > 1:
#                         consolidation_opportunities.append(current_group)
#                     current_group = [current_order['ID']]
        
#         if len(current_group) > 1:
#             consolidation_opportunities.append(current_group)
    
#     return consolidation_opportunities

# # Apply the function to both train and test datasets
# train_consolidation_opportunities_corrected = find_consolidation_opportunities_corrected(train_df)
# test_consolidation_opportunities_corrected = find_consolidation_opportunities_corrected(test_df)

# # Display the consolidation opportunities
# train_consolidation_opportunities_corrected, test_consolidation_opportunities_corrected


In [ ]:
# def find_consolidation_opportunities_fixed(df, eps=0.05, min_samples=2, time_window='1H'):
#     # Convert the delivery locations to a NumPy array
#     locations = df[['Delivery_location_latitude', 'Delivery_location_longitude']].to_numpy()
    
#     # Apply DBSCAN clustering based on location coordinates
#     clustering = DBSCAN(eps=eps, min_samples=min_samples).fit(locations)
#     df['cluster'] = clustering.labels_
    
#     # Initialize an empty list to store consolidation opportunities
#     consolidation_opportunities = []
    
#     # Iterate over each cluster
#     for cluster_id in set(clustering.labels_):
#         if cluster_id == -1:
#             continue
#         cluster_df = df[df['cluster'] == cluster_id].sort_values(by='DateTime_Order_picked')
        
#         # Check for orders within the specified time window
#         current_group = []
#         for i in range(len(cluster_df)):
#             current_order = cluster_df.iloc[i]
#             if not current_group:
#                 current_group.append(current_order['ID'])
#             else:
#                 previous_order = cluster_df.loc[cluster_df['ID'] == current_group[-1]].iloc[0]  # Reference the previous order in the group
#                 time_diff = (current_order['DateTime_Order_picked'] - previous_order['DateTime_Order_picked']).total_seconds() / 3600
#                 if time_diff <= pd.to_timedelta(time_window).total_seconds() / 3600:
#                     current_group.append(current_order['ID'])
#                 else:
#                     if len(current_group) > 1:
#                         consolidation_opportunities.append(current_group)
#                     current_group = [current_order['ID']]
        
#         if len(current_group) > 1:
#             consolidation_opportunities.append(current_group)
    
#     return consolidation_opportunities

# # Apply the function to both train and test datasets
# train_consolidation_opportunities_fixed = find_consolidation_opportunities_fixed(train_df)
# test_consolidation_opportunities_fixed = find_consolidation_opportunities_fixed(test_df)

# # Display the consolidation opportunities
# train_consolidation_opportunities_fixed, test_consolidation_opportunities_fixed


In [ ]:
from sklearn.cluster import DBSCAN
import numpy as np

# Optimized function to find consolidation opportunities using clustering
def find_consolidation_opportunities_optimized(df, eps=0.05, min_samples=2, time_window='1H'):
    # Convert the delivery locations to a NumPy array
    locations = df[['Delivery_location_latitude', 'Delivery_location_longitude']].to_numpy()
    
    # Apply DBSCAN clustering based on location coordinates
    clustering = DBSCAN(eps=eps, min_samples=min_samples).fit(locations)
    df['cluster'] = clustering.labels_
    
    # Initialize an empty list to store consolidation opportunities
    consolidation_opportunities = []
    
    # Iterate over each cluster
    for cluster_id in set(clustering.labels_):
        if cluster_id == -1:
            continue
        cluster_df = df[df['cluster'] == cluster_id].sort_values(by='DateTime_Order_picked')
        
        # Check for orders within the specified time window
        current_group = []
        for i in range(len(cluster_df)):
            current_order = cluster_df.iloc[i]
            if not current_group:
                current_group.append(current_order['ID'])
            else:
                previous_order = cluster_df.loc[cluster_df['ID'] == current_group[-1]].iloc[0]  # Reference the previous order in the group
                time_diff = (current_order['DateTime_Order_picked'] - previous_order['DateTime_Order_picked']).total_seconds() / 3600
                if time_diff <= pd.to_timedelta(time_window).total_seconds() / 3600:
                    current_group.append(current_order['ID'])
                else:
                    if len(current_group) > 1:
                        consolidation_opportunities.append(current_group)
                    current_group = [current_order['ID']]
        
        if len(current_group) > 1:
            consolidation_opportunities.append(current_group)
    
    return consolidation_opportunities

# Apply the optimized function to both train and test datasets
train_df['Order_Date'] = pd.to_datetime(train_df['Order_Date'], format='%d-%m-%Y')
train_df['Time_Order_picked'] = pd.to_datetime(train_df['Time_Order_picked'], format='%H:%M:%S').dt.time
train_df['DateTime_Order_picked'] = pd.to_datetime(train_df['Order_Date'].astype(str) + ' ' + train_df['Time_Order_picked'].astype(str))

test_df['Order_Date'] = pd.to_datetime(test_df['Order_Date'], format='%d-%m-%Y')
test_df['Time_Order_picked'] = pd.to_datetime(test_df['Time_Order_picked'], format='%H:%M:%S').dt.time
test_df['DateTime_Order_picked'] = pd.to_datetime(test_df['Order_Date'].astype(str) + ' ' + test_df['Time_Order_picked'].astype(str))

train_consolidation_opportunities_optimized = find_consolidation_opportunities_optimized(train_df)
test_consolidation_opportunities_optimized = find_consolidation_opportunities_optimized(test_df)

# Display the consolidation opportunities
train_consolidation_opportunities_optimized, test_consolidation_opportunities_optimized


In [ ]:
# Define a function to clean and structure the consolidation opportunities
def clean_format_consolidation_opportunities(df, consolidation_opportunities):
    formatted_data = []

    for index, row in consolidation_opportunities.iterrows():
        order_ids = row['Opportunities'].strip("[]").split(', ')
        if len(order_ids) > 1:
            for i in range(len(order_ids) - 1):
                formatted_data.append({
                    'Order 1': order_ids[i],
                    'Order 2': order_ids[i+1],
                    'Combined Pallets': np.nan,  # Placeholder for Combined Pallets if needed
                    'Distance (km)': np.nan      # Placeholder for Distance if needed
                })
    
    return pd.DataFrame(formatted_data)

# Apply the function to both train and test consolidation opportunities
train_consolidation_cleaned = clean_format_consolidation_opportunities(train_df, train_consolidation_opportunities)
test_consolidation_cleaned = clean_format_consolidation_opportunities(test_df, test_consolidation_opportunities)

# Save the cleaned data to Excel files
train_consolidation_cleaned.to_excel(os.path.join(DATA_DIR, 'train_consolidation_cleaned.xlsx'), index=False)
test_consolidation_cleaned.to_excel(os.path.join(DATA_DIR, 'test_consolidation_cleaned.xlsx'), index=False)

# Provide file paths for download
train_file_path_cleaned = os.path.join(DATA_DIR, 'train_consolidation_cleaned.xlsx')
test_file_path_cleaned = os.path.join(DATA_DIR, 'test_consolidation_cleaned.xlsx')

(train_file_path_cleaned, test_file_path_cleaned)


In [ ]:
import pandas as pd
from sklearn.cluster import DBSCAN
import numpy as np

# Load the provided datasets
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train_df.csv'))
test_df = pd.read_csv(os.path.join(DATA_DIR, 'test_df.csv'))
train_consolidation_opportunities = pd.read_excel(os.path.join(DATA_DIR, 'train_consolidation_opportunities_optimized.xlsx'))
test_consolidation_opportunities = pd.read_excel(os.path.join(DATA_DIR, 'test_consolidation_opportunities_optimized.xlsx'))

# Convert relevant columns to datetime
train_df['Order_Date'] = pd.to_datetime(train_df['Order_Date'], format='%d-%m-%Y')
train_df['Time_Order_picked'] = pd.to_datetime(train_df['Time_Order_picked'], format='%H:%M:%S').dt.time
train_df['DateTime_Order_picked'] = pd.to_datetime(train_df['Order_Date'].astype(str) + ' ' + train_df['Time_Order_picked'].astype(str))

test_df['Order_Date'] = pd.to_datetime(test_df['Order_Date'], format='%d-%m-%Y')
test_df['Time_Order_picked'] = pd.to_datetime(test_df['Time_Order_picked'], format='%H:%M:%S').dt.time
test_df['DateTime_Order_picked'] = pd.to_datetime(test_df['Order_Date'].astype(str) + ' ' + test_df['Time_Order_picked'].astype(str))

# Assuming a column that indicates pallets or using an alternative column for computing 'total_full_pallets'
# Replace 'full_pallets' with an appropriate column if necessary
if 'total_full_pallets' not in train_df.columns:
    if 'some_alternative_column' in train_df.columns:  # Replace with the actual column name if available
        train_df['total_full_pallets'] = train_df['some_alternative_column']
    else:
        train_df['total_full_pallets'] = train_df['Time_taken(min)']  # Example alternative

if 'total_full_pallets' not in test_df.columns:
    if 'some_alternative_column' in test_df.columns:  # Replace with the actual column name if available
        test_df['total_full_pallets'] = test_df['some_alternative_column']
    else:
        test_df['total_full_pallets'] = test_df['Time_taken(min)']  # Example alternative

# Optimized function to find consolidation opportunities using clustering
def find_consolidation_opportunities_optimized(df, eps=0.05, min_samples=2, time_window='1H'):
    locations = df[['Delivery_location_latitude', 'Delivery_location_longitude']].to_numpy()
    clustering = DBSCAN(eps=eps, min_samples=min_samples).fit(locations)
    df['cluster'] = clustering.labels_
    
    consolidation_opportunities = []
    
    for cluster_id in set(clustering.labels_):
        if cluster_id == -1:
            continue
        cluster_df = df[df['cluster'] == cluster_id].sort_values(by='DateTime_Order_picked')
        current_group = []
        for i in range(len(cluster_df)):
            current_order = cluster_df.iloc[i]
            if not current_group:
                current_group.append(current_order['ID'])
            else:
                previous_order = cluster_df.loc[cluster_df['ID'] == current_group[-1]].iloc[0]
                time_diff = (current_order['DateTime_Order_picked'] - previous_order['DateTime_Order_picked']).total_seconds() / 3600
                if time_diff <= pd.to_timedelta(time_window).total_seconds() / 3600:
                    combined_pallets = current_order['total_full_pallets'] + previous_order['total_full_pallets']
                    distance = ((current_order['Delivery_location_latitude'] - previous_order['Delivery_location_latitude'])**2 + 
                                (current_order['Delivery_location_longitude'] - previous_order['Delivery_location_longitude'])**2)**0.5
                    consolidation_opportunities.append({
                        'Order 1': current_order['ID'],
                        'Order 2': previous_order['ID'],
                        'Combined Pallets': combined_pallets,
                        'Distance (km)': distance
                    })
    return consolidation_opportunities

# Apply the function to both train and test datasets
train_consolidation_opportunities_optimized = find_consolidation_opportunities_optimized(train_df)
test_consolidation_opportunities_optimized = find_consolidation_opportunities_optimized(test_df)

# Function to create a cleaner format for the consolidation opportunities
def create_clean_format(consolidation_opportunities):
    formatted_data = []
    for opportunity in consolidation_opportunities:
        formatted_data.append({
            'Order 1': opportunity['Order 1'],
            'Order 2': opportunity['Order 2'],
            'Combined Pallets': opportunity['Combined Pallets'],
            'Distance (km)': opportunity['Distance (km)']
        })
    return pd.DataFrame(formatted_data)

# Convert the opportunities to a cleaner format
train_consolidation_cleaned = create_clean_format(train_consolidation_opportunities_optimized)
test_consolidation_cleaned = create_clean_format(test_consolidation_opportunities_optimized)

# Save the cleaned data to Excel files
train_consolidation_cleaned.to_excel(os.path.join(DATA_DIR, 'train_consolidation_cleaned.xlsx'), index=False)
test_consolidation_cleaned.to_excel(os.path.join(DATA_DIR, 'test_consolidation_cleaned.xlsx'), index=False)




In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Convert opportunities to DataFrame for better visualization
def prepare_opportunities_df(opportunities, order_df):
    opportunities_list = []
    for group in opportunities:
        if len(group) > 1:
            for i in range(len(group)-1):
                order1 = order_df[order_df['ID'] == group[i]].iloc[0]
                order2 = order_df[order_df['ID'] == group[i+1]].iloc[0]
                distance = ((order1['Delivery_location_latitude'] - order2['Delivery_location_latitude'])**2 + 
                            (order1['Delivery_location_longitude'] - order2['Delivery_location_longitude'])**2)**0.5
                combined_pallets = order1['total_full_pallets'] + order2['total_full_pallets']
                opportunities_list.append((group[i], group[i+1], combined_pallets, distance))
    
    opportunities_df = pd.DataFrame(opportunities_list, columns=['Order 1', 'Order 2', 'Combined Pallets', 'Distance (km)'])
    opportunities_df['Order Pair'] = opportunities_df['Order 1'].astype(str) + " & " + opportunities_df['Order 2'].astype(str)
    return opportunities_df

# Prepare the dataframes
train_opportunities_df = prepare_opportunities_df(os.path.join(DATA_DIR, 'train_consolidation_opportunities_optimized.xlsx'), train_df)
test_opportunities_df = prepare_opportunities_df(os.path.join(DATA_DIR, 'test_consolidation_opportunities_optimized.xlsx'), test_df)

# Function to visualize data distributions and relationships
def visualize_data(order_df, opportunities_df):
    plt.figure(figsize=(14, 7))
    
    # Scatter plot of orders with consolidation opportunities
    plt.subplot(1, 2, 1)
    plt.scatter(order_df['Delivery_location_longitude'], order_df['Delivery_location_latitude'], c='blue', label='Orders', alpha=0.5)
    for _, row in opportunities_df.iterrows():
        order1 = order_df[order_df['ID'] == row['Order 1']].iloc[0]
        order2 = order_df[order_df['ID'] == row['Order 2']].iloc[0]
        plt.plot([order1['Delivery_location_longitude'], order2['Delivery_location_longitude']], 
                 [order1['Delivery_location_latitude'], order2['Delivery_location_latitude']], 'ro-')
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.title('Scatter Plot of Orders with Consolidation Opportunities')
    plt.legend()

    # Bar plot of combined pallets for top consolidation opportunities
    top_opportunities_df = opportunities_df.sort_values(by='Combined Pallets', ascending=False).head(20)
    plt.subplot(1, 2, 2)
    sns.barplot(data=top_opportunities_df, x='Order Pair', y='Combined Pallets', palette='viridis')
    plt.xticks(rotation=45, ha='right')
    plt.title('Top 20 Combined Pallets for Consolidation Opportunities')
    plt.xlabel('Order Pairs')
    plt.ylabel('Combined Pallets')

    plt.tight_layout()
    plt.show()

# Visualize the data for train and test datasets
visualize_data(train_df, train_opportunities_df)
visualize_data(test_df, test_opportunities_df)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Function to prepare opportunities DataFrame
def prepare_opportunities_df(opportunities, order_df):
    opportunities_list = []
    for group in opportunities:
        if len(group) > 1:
            for i in range(len(group)-1):
                order1 = order_df[order_df['ID'] == group[i]].iloc[0]
                order2 = order_df[order_df['ID'] == group[i+1]].iloc[0]
                distance = ((order1['Delivery_location_latitude'] - order2['Delivery_location_latitude'])**2 + 
                            (order1['Delivery_location_longitude'] - order2['Delivery_location_longitude'])**2)**0.5
                combined_pallets = order1['multiple_deliveries'] + order2['multiple_deliveries']  # Adjust column for pallets
                opportunities_list.append((group[i], group[i+1], combined_pallets, distance))
    
    opportunities_df = pd.DataFrame(opportunities_list, columns=['Order 1', 'Order 2', 'Combined Pallets', 'Distance (km)'])
    opportunities_df['Order Pair'] = opportunities_df['Order 1'].astype(str) + " & " + opportunities_df['Order 2'].astype(str)
    return opportunities_df

# Prepare the dataframes
train_opportunities_df = prepare_opportunities_df(train_consolidation_opportunities_optimized, train_df)
test_opportunities_df = prepare_opportunities_df(test_consolidation_opportunities_optimized, test_df)

# Function to visualize scatter plot
def visualize_scatter_plot(order_df, opportunities_df):
    plt.figure(figsize=(10, 6))
    plt.scatter(order_df['Delivery_location_longitude'], order_df['Delivery_location_latitude'], c='blue', label='Orders', alpha=0.5)
    for _, row in opportunities_df.iterrows():
        order1 = order_df[order_df['ID'] == row['Order 1']].iloc[0]
        order2 = order_df[order_df['ID'] == row['Order 2']].iloc[0]
        plt.plot([order1['Delivery_location_longitude'], order2['Delivery_location_longitude']], 
                 [order1['Delivery_location_latitude'], order2['Delivery_location_latitude']], 'ro-')
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.title('Scatter Plot of Orders with Consolidation Opportunities')
    plt.legend()
    plt.show()

# Function to visualize bar plot
def visualize_bar_plot(opportunities_df):
    top_opportunities_df = opportunities_df.sort_values(by='Combined Pallets', ascending=False).head(20)
    plt.figure(figsize=(14, 8))
    sns.barplot(data=top_opportunities_df, x='Order Pair', y='Combined Pallets', palette='viridis')
    plt.xticks(rotation=45, ha='right')
    plt.title('Top 20 Combined Pallets for Consolidation Opportunities')
    plt.xlabel('Order Pairs')
    plt.ylabel('Combined Pallets')
    plt.tight_layout()
    plt.show()

# Visualize scatter plot for train and test datasets
visualize_scatter_plot(train_df, train_opportunities_df)
visualize_scatter_plot(test_df, test_opportunities_df)

# Visualize bar plot for train and test datasets
visualize_bar_plot(train_opportunities_df)
visualize_bar_plot(test_opportunities_df)
